In [ ]:
pip install -U akshare pandas pyarrow tqdm

In [2]:
import os, pathlib
print("当前工作目录 =", os.getcwd())
print("目录内容 =", list(pathlib.Path().iterdir()))


当前工作目录 = C:\Users\tao11\Downloads\Quant Research\debts\data
目录内容 = [WindowsPath('.ipynb_checkpoints'), WindowsPath('cb_SH_full.parquet'), WindowsPath('cb_SZ_full.parquet'), WindowsPath('fetch_cb_data.py'), WindowsPath('Untitled.ipynb')]


In [11]:
# fetch_all_cb.py
import akshare as ak, pandas as pd, datetime as dt, time, random, os
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

ROOT   = Path(".")
OUT_SH = ROOT / "cb_SH_full.parquet"
OUT_SZ = ROOT / "cb_SZ_full.parquet"
MAX_WORKERS = 6          # 线程数，根据网络情况调

# A. 拿最新代码表
spot = ak.bond_zh_hs_cov_spot()
codes_all = spot["symbol"].astype(str)

# 允许增量：若文件已存在就先读出来
def load_existing(path):
    if path.exists() and path.stat().st_size > 0:
        return pd.read_parquet(path)
    return pd.DataFrame()

df_sh_exist = load_existing(OUT_SH)
df_sz_exist = load_existing(OUT_SZ)
codes_done  = set(df_sh_exist["symbol"]).union(df_sz_exist["symbol"])

codes_todo  = [c for c in codes_all if c not in codes_done]
print(f"📝 需要下载 {len(codes_todo)} / {len(codes_all)} 只")

# B. 单只下载函数，自动回退到 bond_zh_cov_daily（兼容退市）
def fetch_one(code):
    try:
        df = ak.bond_zh_hs_cov_daily(symbol=code)
        if df.empty:
            df = ak.bond_zh_cov_daily(symbol=code)   # 退市或历史久远
        if df.empty:
            return None
        df["date"]   = pd.to_datetime(df["date"])
        df["symbol"] = code
        return df
    except Exception as e:
        print(f"⚠️  {code} 失败：{e}")
        return None

# C. 并发抓取
frames_sh, frames_sz = [], []
with ThreadPoolExecutor(MAX_WORKERS) as pool:
    futures = {pool.submit(fetch_one, c): c for c in codes_todo}
    for fut in as_completed(futures):
        code = futures[fut]
        df   = fut.result()
        if df is None: 
            continue
        (frames_sh if code.startswith("sh") else frames_sz).append(df)
        # 随机 sleep，避免触发反爬
        time.sleep(random.uniform(0.2, 0.6))

# D. 合并旧数据 & 写入 Parquet（覆盖写）
def concat_save(frames_new, df_exist, out_path):
    if frames_new or not df_exist.empty:
        df_all = pd.concat([df_exist, *frames_new], ignore_index=True)
        df_all.to_parquet(out_path, index=False)
        print(f"✅  写入 {out_path}，尺寸：{df_all.shape}")

concat_save(frames_sh, df_sh_exist, OUT_SH)
concat_save(frames_sz, df_sz_exist, OUT_SZ)


  0%|          | 0/6 [00:00<?, ?it/s]

📝 需要下载 480 / 480 只
⚠️  sh110801 失败：'date'
⚠️  sh110808 失败：'date'
⚠️  sh110807 失败：'date'
⚠️  sh110809 失败：'date'
⚠️  sh110805 失败：'date'
⚠️  sh110815 失败：'date'
⚠️  sh110811 失败：'date'
⚠️  sh110817 失败：'date'
⚠️  sh110816 失败：'date'
⚠️  sh118500 失败：'date'
✅  写入 cb_SH_full.parquet，尺寸：(151888, 8)
✅  写入 cb_SZ_full.parquet，尺寸：(192557, 8)


In [12]:
# check_cb_dataset.py
import pandas as pd, akshare as ak, pyarrow, datetime as dt
from pathlib import Path
from tqdm import tqdm

ROOT = Path(".")                 # 或 Path.cwd()
FILES = {
    "sh": ROOT / "cb_SH_full.parquet",
    "sz": ROOT / "cb_SZ_full.parquet",
}
# 1) 读入 Parquet
dfs = {ex: pd.read_parquet(path) for ex, path in FILES.items()}

# 2) 快速数据健康检查
def describe(df):
    if df.empty:
        return pd.DataFrame(
            columns=["start", "end", "rows", "days", "missing_ratio"]
        )
    g = df.groupby("symbol", observed=True)
    res = (g.agg(start=("date", "min"),
                 end  =("date", "max"),
                 rows =("date", "size"))
             .assign(days=lambda x: (x["end"]-x["start"]).dt.days+1,
                     missing_ratio=lambda x: 1-x["rows"]/x["days"])
             .sort_values("start"))
    return res


stat_sh, stat_sz = map(describe, dfs.values())

# 3) 拿“应有”代码清单
spot = ak.bond_zh_hs_cov_spot()           # 需联网
code_col = "symbol" if "symbol" in spot else "bond_code"
all_codes_spot = spot[code_col].astype(str)

# 4) 对比集合
have   = pd.Index(stat_sh.index.tolist()+stat_sz.index.tolist())
should = pd.Index(all_codes_spot.tolist())

missing = should.difference(have)         # 理论上应有但没在文件里的
extra   = have.difference(should)         # 文件里有但已退市／接口未列出的

# 5) 打印报告
print("="*60)
print(" 可转债数据覆盖情况 (沪+深) ")
print("  截止：", dt.date.today())
print("="*60)
print(f"应有标的数（交易所现存+退市?）: {len(should):>6}")
print(f"文件包含标的数               : {len(have):>6}")
print(f"缺失标的数                   : {len(missing):>6}")
print(f"多余/退市标的数              : {len(extra):>6}\n")

if len(missing):
    print("⚠️  缺失列表（前 20）：", missing[:20].tolist())

# 6) 查异常数据长度
print("\n--- 上海 ---")
print(stat_sh.describe().loc[["min","max"]].T)
abnormal_sh = stat_sh.query("rows < 250")     # 少于 ~1 年
print("⚠️  行数极少(可能没抓到) ：", abnormal_sh.index.tolist()[:10])

print("\n--- 深圳 ---")
print(stat_sz.describe().loc[["min","max"]].T)
abnormal_sz = stat_sz.query("rows < 250")
print("⚠️  行数极少(可能没抓到) ：", abnormal_sz.index.tolist()[:10])


  0%|          | 0/6 [00:00<?, ?it/s]

 可转债数据覆盖情况 (沪+深) 
  截止： 2025-07-17
应有标的数（交易所现存+退市?）:    480
文件包含标的数               :    470
缺失标的数                   :     10
多余/退市标的数              :      0

⚠️  缺失列表（前 20）： ['sh110801', 'sh110805', 'sh110807', 'sh110808', 'sh110809', 'sh110811', 'sh110815', 'sh110816', 'sh110817', 'sh118500']

--- 上海 ---
                               min                  max
start          2019-08-23 00:00:00  2025-07-16 00:00:00
end            2024-07-17 00:00:00  2025-07-17 00:00:00
rows                           2.0               1429.0
days                           2.0               2156.0
missing_ratio                  0.0             0.350388
⚠️  行数极少(可能没抓到) ： ['sh113685', 'sh118048', 'sh113686', 'sh113687', 'sh111020', 'sh111021', 'sh118049', 'sh110096', 'sh118050', 'sh113689']

--- 深圳 ---
                               min                  max
start          2018-09-04 00:00:00  2025-07-04 00:00:00
end            2021-06-28 00:00:00  2025-07-17 00:00:00
rows                          10.0      